# Figure S6: Posterior pairplot

This notebook reads the retained posterior samples and writes Figure S6 beside the other visualization notebooks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sbi.analysis import pairplot

_here = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in [_here, *_here.parents]
     if (path / "6_metapopulation_model" / "data").is_dir()),
    _here,
)
INFERENCE_DIR = REPO_ROOT / "6_metapopulation_model" / "outputs" / "inference"
FIGURES_DIR = REPO_ROOT / "6_metapopulation_model" / "visualization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
posterior_path = INFERENCE_DIR / "posterior_samples.csv"
pairplot_path = FIGURES_DIR / "Figure S6_pairplot.png"

if not posterior_path.is_file():
    raise FileNotFoundError(f"Input file not found: {posterior_path}")


In [ ]:
posterior_df = pd.read_csv(posterior_path)
param_names = list(posterior_df.columns)

PARAM_LABELS = [
    r"$\beta_{\mathrm{intra}}^{\mathrm{CS}}$",
    r"$\beta_{\mathrm{intra}}^{\mathrm{FM}}$",
    r"$\beta_{\mathrm{intra}}^{\mathrm{Oth}}$",
    r"$\beta_{\mathrm{intra}}^{\mathrm{RB}}$",
    r"$\beta_{\mathrm{local}}^{\mathrm{CS}}$",
    r"$\beta_{\mathrm{local}}^{\mathrm{FM}}$",
    r"$\beta_{\mathrm{local}}^{\mathrm{Oth}}$",
    r"$\beta_{\mathrm{local}}^{\mathrm{RB}}$",
    r"$\beta_{\mathrm{inter}}$",
    r"$k$",
    r"$\rho$",
]
if len(param_names) != len(PARAM_LABELS):
    raise ValueError(
        f"Expected {len(PARAM_LABELS)} posterior parameters, "
        f"found {len(param_names)}: {param_names}"
    )

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

samples = torch.as_tensor(posterior_df[param_names].to_numpy(), dtype=torch.float32)
medians = torch.as_tensor(
    posterior_df[param_names].median().to_numpy(), dtype=torch.float32
).reshape(1, -1)
fig, _ = pairplot(
    samples,
    points=medians,
    limits=None,
    upper="hist",
    diag="hist",
    figsize=(16, 16),
    labels=PARAM_LABELS,
    diag_kwargs={"bins": 50, "density": True},
    upper_kwargs={"bins": 50},
)
fig.tight_layout()
fig.savefig(pairplot_path, bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved: {pairplot_path}")
